In [1]:
import sys
sys.path.append('../../')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

import pandas as pd
from Bio import Entrez

Entrez.email = 'elisa.m.zavala@ntnu.no'

In [2]:
INPUT_DIR

'/Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/files/Input'

In [3]:
# Load the ChEBI database into a Pandas DataFrame.

full_compounds_file = INPUT_DIR+'/Products/compounds.tsv'
full_compounds = pd.read_table(full_compounds_file, dtype=str, index_col='ID')
full_compounds = full_compounds.loc[:,'NAME']
full_compounds.head()

ID
9349        sulfonyldimethane
9352                 sulindac
9355               sulfuretin
9380                 syringin
9427    2-methylanthraquinone
Name: NAME, dtype: object

In [4]:
full_compounds.shape

(189584,)

In [5]:
# Load the ChEBI database into a Pandas DataFrame.

full_syn_file = INPUT_DIR+'/Products/names.tsv'
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='COMPOUND_ID')
full_syn = full_syn.loc[full_syn.LANGUAGE=='en',['TYPE','NAME']]
full_syn.head()

,TYPE,NAME
COMPOUND_ID,,
16478,SYNONYM,"N-Acetyl-beta-D-glucosaminyl-1,6-(N-acetyl-bet..."
15947,SYNONYM,N-Acetyl-beta-D-glucosaminylamine
7853,SYNONYM,Oxyacanthine
15379,SYNONYM,Oxygen
15379,SYNONYM,O2


In [42]:
full_syn = pd.read_table(full_syn_file, dtype=str, index_col='COMPOUND_ID')


In [6]:
full_syn.shape

(337398, 2)

In [7]:
duplicates = full_syn.index[full_syn.index.duplicated()]
print(full_syn.loc[duplicates])


                   TYPE                                       NAME
COMPOUND_ID                                                       
15379           SYNONYM                                     Oxygen
15379           SYNONYM                                         O2
15379        IUPAC NAME                                   dioxygen
15379           SYNONYM                           molecular oxygen
15379           SYNONYM                                         O2
...                 ...                                        ...
60330           SYNONYM                                        ADO
60330           SYNONYM  2-methyl-2-(methylsulfanyl)propanaldoxime
60330              NAME  2-methyl-2-(methylsulfanyl)propanal oxime
60197           SYNONYM                                     BChl c
60197              NAME                    a bacteriochlorophyll c

[1977078 rows x 2 columns]


In [8]:
full_syn = full_syn.drop_duplicates()
full_syn = full_syn.groupby('COMPOUND_ID').apply(lambda x: x['NAME'].tolist())
full_syn.name = 'Synonym'

In [9]:
ChEBI = pd.concat([full_compounds,full_syn], axis=1)

In [10]:
ChEBI = ChEBI.dropna(how='all')

In [11]:
ChEBI.loc[17234].Synonym

['Glucose', 'glucose', 'Glukose', 'Glc', 'gluco-hexose', 'DL-glucose']

In [12]:
ChEBI.shape

(170158, 2)

In [18]:
ChEBI.dropna(subset=['Synonym']).to_json('rm_chebi.json')